# UC-02: Invoice Three-Way Match — Inference Demo

Demonstrates the trained model in inference mode:
- Single invoice predictions with explanations
- Batch scoring with risk ranking
- SHAP-based natural language explanations

## 1. Load Model & Feature Pipeline

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

from ml.common.db_config import load_tables
from ml.data_processing.python.uc02_preprocessing import UC02_TABLES
from ml.uc_02_invoice_match.inference.serve import InvoiceMatchPredictor

In [ ]:
# Load model
model_path = project_root / "ml" / "uc_02_invoice_match" / "training" / "best_model.joblib"
predictor = InvoiceMatchPredictor(model_path)
print(f"Model: {predictor.model_name}")
print(f"Target: {predictor.target}")
print(f"Features: {len(predictor.feature_columns)}")

# Load data tables
csv_dir = project_root / "output" / "csv"
tables = load_tables("csv", UC02_TABLES, csv_dir=csv_dir)
print(f"\nLoaded {len(tables)} tables")

## 2. Simulate New Invoice Arrival

We pick 5 invoices as if they just arrived and need to be scored.

In [ ]:
# Select 5 diverse invoices
invoice_header = tables["invoice_header"]
sample_invoices = (
    invoice_header
    .groupby("match_status")
    .apply(lambda g: g.sample(min(2, len(g)), random_state=42), include_groups=False)
    .reset_index(drop=True)
    .head(5)
)

print("Sample invoices for scoring:")
print(sample_invoices[["invoice_id", "vendor_id", "invoice_date", "match_status"]].to_string(index=False))

## 3. Single Invoice Predictions

In [ ]:
for _, inv_row in sample_invoices.iterrows():
    inv_id = inv_row["invoice_id"]
    actual = inv_row["match_status"]

    result = predictor.predict({"invoice_id": inv_id}, tables)

    print(f"\n{'='*60}")
    print(f"Invoice: {inv_id}")
    print(f"  Actual:     {actual}")
    print(f"  Predicted:  {result['predicted_class']} ({result['probability']:.1%})")
    print(f"  Confidence: {result['confidence']}")
    print(f"  P(variance): {result['class_probabilities'].get('ANY_VARIANCE', 'N/A')}")
    if result["top_features"]:
        print(f"  Top features:")
        for feat in result["top_features"][:3]:
            print(f"    - {feat['feature']}: {feat['value']:.3f} (importance={feat['importance']:.3f})")

## 4. SHAP Explanations

In [ ]:
# Build features for SHAP (inference mode — no LOO)
from ml.uc_02_invoice_match.feature_engineering.feature_functions import (
    build_uc02_features, prepare_feature_matrix,
)

feature_df = build_uc02_features(tables, leave_one_out=False)
X, y = prepare_feature_matrix(feature_df, target="binary")

# Align columns
missing_cols = set(predictor.feature_columns) - set(X.columns)
for col in missing_cols:
    X[col] = 0
X = X[predictor.feature_columns]

# SHAP explainer
model = predictor.model
if hasattr(model, "feature_importances_"):
    explainer = shap.TreeExplainer(model)
else:
    background = shap.sample(X, 50)
    explainer = shap.KernelExplainer(model.predict_proba, background)

In [ ]:
# Per-invoice SHAP explanation with natural language template
sample_ids = sample_invoices["invoice_id"].tolist()
sample_mask = feature_df["invoice_id"].isin(sample_ids)
sample_indices = feature_df.index[sample_mask].tolist()

for inv_id in sample_ids[:3]:
    inv_mask = feature_df["invoice_id"] == inv_id
    if not inv_mask.any():
        continue

    idx = inv_mask[inv_mask].index[0]
    pos = list(X.index).index(idx)

    shap_vals = explainer.shap_values(X.iloc[[pos]])
    if isinstance(shap_vals, list):
        sv = shap_vals[1][0]
    else:
        sv = shap_vals[0]

    proba = model.predict_proba(X.iloc[[pos]])[0]
    top_idx = np.argsort(np.abs(sv))[::-1][:5]

    risk_level = "HIGH" if proba[1] > 0.5 else "LOW"
    print(f"\n{'='*70}")
    print(f"Invoice {inv_id}: {risk_level} RISK ({proba[1]:.1%} variance probability)")
    print(f"\nThis invoice is predicted to {'have matching issues' if proba[1] > 0.5 else 'match cleanly'} because:")
    for rank, i in enumerate(top_idx, 1):
        feat = predictor.feature_columns[i]
        val = X.iloc[pos, i]
        direction = "increases" if sv[i] > 0 else "decreases"
        print(f"  {rank}. {feat} = {val:.2f} → {direction} risk (SHAP = {sv[i]:+.3f})")

## 5. Batch Scoring

In [ ]:
# Score all invoices
batch_results = predictor.predict_batch(tables)

print(f"Scored {len(batch_results)} invoices")
print(f"\nPrediction distribution:")
print(batch_results["predicted_class"].value_counts())

print(f"\nTop 10 highest-risk invoices:")
print(batch_results.head(10)[["invoice_id", "predicted_class", "probability", "variance_probability"]].to_string(index=False))

In [ ]:
# Compare predictions with actuals
actuals = tables["invoice_header"][["invoice_id", "match_status"]].copy()
actuals["actual_binary"] = (actuals["match_status"] != "FULL_MATCH").astype(int)

comparison = batch_results.merge(actuals, on="invoice_id", how="left")
comparison["pred_binary"] = (comparison["predicted_class"] != "FULL_MATCH").astype(int)

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(comparison["actual_binary"], comparison["pred_binary"])
ConfusionMatrixDisplay(cm, display_labels=["FULL_MATCH", "ANY_VARIANCE"]).plot(ax=ax)
ax.set_title("Batch Scoring — Confusion Matrix")
plt.tight_layout()
plt.show()

# Accuracy
accuracy = (comparison["actual_binary"] == comparison["pred_binary"]).mean()
print(f"\nBatch accuracy: {accuracy:.1%}")

## 6. Risk Distribution

In [ ]:
# Variance probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(batch_results["variance_probability"], bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(0.5, color="red", linestyle="--", label="Decision threshold")
axes[0].set_title("Variance Probability Distribution")
axes[0].set_xlabel("P(variance)")
axes[0].legend()

# Sorted risk curve
sorted_probs = batch_results["variance_probability"].sort_values(ascending=False).values
axes[1].plot(range(len(sorted_probs)), sorted_probs, color="coral")
axes[1].axhline(0.5, color="red", linestyle="--", label="Decision threshold")
axes[1].set_title("Invoices Ranked by Variance Risk")
axes[1].set_xlabel("Invoice Rank")
axes[1].set_ylabel("P(variance)")
axes[1].legend()

plt.tight_layout()
plt.show()